# Assignment Case B — Vendor Performance
## Working with PySpark DataFrames for Analytics
**Day 42 | Batch 13 | 03 Mei 2026**

---

| Field | Isi |
|-------|-----|
| **Nama** | *Muhammad Qois Huzyan Octava* |
| **NIM** | *(isi NIM kamu)* |
| **Case** | Case B — Procurement & Vendor Performance |
| **Source** | Hive: `adventureworks.*` |
| **Target** | Hive: `adventureworks_curated.fact_vendor_performance` |

---

## ⚠️ Filter Penting

> **Kamu HANYA boleh menggunakan data dengan ShipMethod: `CARGO TRANSPORT 5` dan `OVERNIGHT J-FAST`.**
> Filter ini harus diterapkan sebelum agregasi.

## 🎯 Business Question

1. **Vendor mana yang paling on-time** dalam pengiriman via CARGO TRANSPORT 5 dan OVERNIGHT J-FAST?
2. **Apakah harga beli dari vendor sudah kompetitif** dibanding StandardCost produk?
3. **Vendor mana yang memiliki VendorScore tertinggi** secara keseluruhan?
4. **Korelasi** antara CreditRating vendor dan OnTimeRate?

## 📐 Formula VendorScore

```
VendorScore = (OnTimeRate * 0.6) + ((100 - ABS(AvgPriceVariance)) * 0.4)
```

- **OnTimeRate** = OnTimeOrders / TotalOrders * 100  (ShipDate <= DueDate)
- **AvgPriceVariance** = rata-rata (UnitPrice - StandardCost) per baris PO detail
- ⚠️ Exclude baris dengan ShipDate IS NULL dari kalkulasi OnTimeRate

## 📋 Kolom Target: `fact_vendor_performance`

| Kolom | Tipe | Keterangan |
|-------|------|------------|
| `vendor_id` | INT | ID vendor |
| `vendor_name` | STRING | Nama vendor |
| `credit_rating` | INT | Rating kredit (1-5) |
| `ship_method` | STRING | Metode pengiriman |
| `total_orders` | INT | Total PO |
| `on_time_orders` | INT | PO dengan ShipDate <= DueDate |
| `on_time_rate` | DECIMAL(5,2) | % ketepatan pengiriman |
| `avg_lead_time_days` | DECIMAL(5,1) | Rata-rata (ShipDate - OrderDate) |
| `avg_price_variance` | DECIMAL(10,2) | Rata-rata (UnitPrice - StandardCost) |
| `vendor_score` | DECIMAL(5,2) | Formula scorecard |
| `load_timestamp` | TIMESTAMP | Waktu load |


## 0. Persiapan: Buat Hive External Tables untuk Vendor Performance

Sebelum mulai analisis, kamu perlu menyiapkan tabel sumber sebagai
**Hive External Tables**. Pola tutorialnya sama dengan:
- `02_adventureworks_to_hdfs.ipynb` → extract PostgreSQL ke HDFS Parquet (dimensi + fakta dengan partisi)
- `03_hdfs_to_hive.ipynb` → buat External Tables di atas Parquet

**Tabel yang akan dibuat di Hive (`adventureworks.*`):**
| Hive Table | Sumber PostgreSQL | Catatan |
|---|---|---|
| `fact_purchase_orders` | `Purchasing.PurchaseOrderHeader` | partisi `order_year`, `order_month` |
| `fact_purchase_details` | `Purchasing.PurchaseOrderDetail` | tanpa partisi |
| `dim_vendor` | `Purchasing.Vendor` | |
| `dim_ship_method` | `Purchasing.ShipMethod` | |
| `dim_product` | `Production.Product` | |

> **Lewati bagian ini** jika tabel sudah ada di Hive.

### 0.1 Setup SparkSession JDBC (untuk Extract ke HDFS)

Mirip dengan `02_adventureworks_to_hdfs.ipynb`. SparkSession ini
**tanpa Hive support** — hanya untuk extract data dari PostgreSQL ke HDFS sebagai Parquet.

In [1]:
# TODO: setup environment & import
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--jars /home/jovyan/work/jars/postgresql-42.7.3.jar pyspark-shell'  # TODO: '--jars /home/jovyan/work/jars/postgresql-42.7.3.jar pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F_jdbc

# TODO: buat spark_jdbc tanpa Hive support
spark_jdbc = SparkSession.builder \
    .master('local[*]') \
    .appName('Vendor-to-HDFS') \
    .config('spark.sql.parquet.compression.codec', 'snappy') \
    .getOrCreate()

# TODO: konfigurasi JDBC PostgreSQL AdventureWorks
JDBC_URL   = 'jdbc:postgresql://adventureworks-postgres:5432/postgres'  # TODO: 'jdbc:postgresql://adventureworks-postgres:5432/postgres'
JDBC_PROPS = {
                    "user": "postgres",
                    "password": "My_password1",
                    "driver": "org.postgresql.Driver"
                }  # TODO: dict berisi 'user', 'password', 'driver'
HDFS_BASE  = 'hdfs://namenode:9000/datalake/raw'  # TODO: 'hdfs://namenode:9000/datalake/raw'

def read_pg(schema, table):
    return spark_jdbc.read.jdbc(
        url=JDBC_URL,
        table=f'"{schema}"."{table}"',
        properties=JDBC_PROPS
    )

print(f'spark_jdbc {spark_jdbc.version} ready')


spark_jdbc 3.5.0 ready


### 0.2 Extract Tabel Dimensi ke HDFS

Tabel dimensi tidak perlu di-partisi (ukurannya kecil).

In [2]:
# TODO: lengkapi list tabel dimensi
dim_tables = [
    ('Purchasing', 'Vendor'),  # TODO: ('Purchasing', 'Vendor')
    ('Purchasing', 'ShipMethod'),  # TODO: ('Purchasing', 'ShipMethod')
    ('Production', 'Product'),  # TODO: ('Production', 'Product')
    ('Purchasing', 'PurchaseOrderDetail')  # TODO: ('Purchasing', 'PurchaseOrderDetail')  -- treat as dim (tanpa partisi)
]

for schema, table in dim_tables:
    df = read_pg(schema, table)  # TODO: read_pg(schema, table)
    hdfs_path = f'{HDFS_BASE}/{schema}/{table}'  # TODO: f'{HDFS_BASE}/{schema}/{table}'
    df.write.mode('overwrite').parquet(hdfs_path)
    print(f'  {schema}.{table:30s} -> {df.count():6,} rows -> HDFS')

print('Tabel dimensi selesai disimpan!')


  Purchasing.Vendor                         ->    104 rows -> HDFS
  Purchasing.ShipMethod                     ->      5 rows -> HDFS
  Production.Product                        ->    504 rows -> HDFS
  Purchasing.PurchaseOrderDetail            ->  8,845 rows -> HDFS
Tabel dimensi selesai disimpan!


### 0.3 Extract Tabel Fakta dengan Partisi

`PurchaseOrderHeader` berukuran besar dan punya kolom `OrderDate`. Partisi
berdasarkan `order_year` dan `order_month` — sama polanya dengan
`SalesOrderHeader` di notebook `02`.

In [3]:
# TODO: read PurchaseOrderHeader dari PostgreSQL
df_poh = read_pg('Purchasing', 'PurchaseOrderHeader')  # TODO: read_pg('Purchasing', 'PurchaseOrderHeader')

# TODO: tambahkan kolom partisi order_year & order_month dari OrderDate
df_poh_part = df_poh \
    .withColumn('order_year',  F_jdbc.year('orderdate')) \
    .withColumn('order_month', F_jdbc.month('orderdate'))  # TODO: F_jdbc.month('orderdate')

po_path = f'{HDFS_BASE}/Purchasing/PurchaseOrderHeader'

# TODO: simpan ke HDFS dengan partisi (order_year, order_month)
df_poh_part.write.mode('overwrite') \
    .partitionBy('order_year', 'order_month') \
    .parquet(po_path)

print(f'  PurchaseOrderHeader -> {df_poh.count():,} rows -> HDFS (partisi order_year/order_month)')


  PurchaseOrderHeader -> 4,012 rows -> HDFS (partisi order_year/order_month)


### 0.4 Setup SparkSession dengan Hive Support

Mirip dengan `03_hdfs_to_hive.ipynb`. Tutup `spark_jdbc` dulu, lalu buat
SparkSession baru **dengan** Hive support untuk membuat External Tables.

In [4]:
# TODO: stop spark_jdbc agar tidak konflik resource
spark_jdbc.stop()

from pyspark.sql import SparkSession

# TODO: buat spark_hive dengan Hive support
spark_hive = SparkSession.builder \
    .master('local[*]') \
    .appName('Vendor-HiveSetup') \
    .config('spark.jars', '/home/jovyan/work/jars/postgresql-42.7.3.jar') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .enableHiveSupport() \
    .getOrCreate()

HDFS_BASE = 'hdfs://namenode:9000/datalake/raw'

# TODO: buat database adventureworks jika belum ada
spark_hive.sql('CREATE DATABASE IF NOT EXISTS adventureworks')  # TODO: CREATE DATABASE IF NOT EXISTS adventureworks
spark_hive.sql('USE adventureworks')
print(f'spark_hive {spark_hive.version} ready')
spark_hive.sql('SHOW DATABASES').show()


spark_hive 3.5.0 ready
+--------------+
|     namespace|
+--------------+
|adventureworks|
|       default|
+--------------+



### 0.5 Buat Hive External Tables — Tabel Dimensi

Tabel dimensi tanpa partisi → schema otomatis di-infer.

In [5]:
# Helper function — sama persis dengan pola di 03_hdfs_to_hive.ipynb
def create_external_table(table_name, hdfs_path):
    spark_hive.sql(f'DROP TABLE IF EXISTS adventureworks.{table_name}')
    spark_hive.sql(f"""
        CREATE EXTERNAL TABLE adventureworks.{table_name}
        USING PARQUET
        LOCATION '{hdfs_path}'
    """)
    count = spark_hive.sql(f'SELECT COUNT(*) FROM adventureworks.{table_name}').collect()[0][0]
    print(f'  {table_name:35s} -> {count:6,} rows')

# TODO: lengkapi mapping nama tabel Hive -> path HDFS
hive_dim_tables = [
    ('dim_vendor',         f'{HDFS_BASE}/Purchasing/Vendor'),  # TODO: ('dim_vendor',         f'{HDFS_BASE}/Purchasing/Vendor')
    ('dim_ship_method',    f'{HDFS_BASE}/Purchasing/ShipMethod'),  # TODO: ('dim_ship_method',    f'{HDFS_BASE}/Purchasing/ShipMethod')
    ('dim_product',        f'{HDFS_BASE}/Production/Product'),  # TODO: ('dim_product',        f'{HDFS_BASE}/Production/Product')
    ('fact_purchase_details', f'{HDFS_BASE}/Purchasing/PurchaseOrderDetail'),  # TODO: ('fact_purchase_details', f'{HDFS_BASE}/Purchasing/PurchaseOrderDetail')
]

for tname, path in hive_dim_tables:
    create_external_table(tname, path)  # TODO

print('Tabel dimensi Hive selesai!')


  dim_vendor                          ->    104 rows
  dim_ship_method                     ->      5 rows
  dim_product                         ->    504 rows
  fact_purchase_details               ->  8,845 rows
Tabel dimensi Hive selesai!


### 0.6 Buat Hive External Table — Tabel Fakta dengan Partisi

Untuk tabel berpartisi, perlu deklarasi `PARTITIONED BY` di DDL
lalu jalankan `MSCK REPAIR TABLE` agar Hive mengenali partisinya.
Pola sama dengan `fact_sales_orders` di `03_hdfs_to_hive.ipynb`.

In [6]:
po_path = f'{HDFS_BASE}/Purchasing/PurchaseOrderHeader'

# TODO: drop dulu kalau ada
spark_hive.sql('DROP TABLE IF EXISTS adventureworks.fact_purchase_orders')  # TODO: 'DROP TABLE IF EXISTS adventureworks.fact_purchase_orders'

# TODO: buat External Table dengan skema eksplisit + PARTITIONED BY
# Petunjuk: kolom partisi (order_year, order_month) JANGAN ditulis di list kolom utama
spark_hive.sql(f"""
    CREATE EXTERNAL TABLE adventureworks.fact_purchase_orders (
        PurchaseOrderID    INT,
        RevisionNumber     SMALLINT,
        Status             SMALLINT,
        EmployeeID         INT,
        VendorID           INT,
        ShipMethodID       INT,
        OrderDate          TIMESTAMP,
        ShipDate           TIMESTAMP,
        DueDate            TIMESTAMP,  -- TODO: tambahkan kolom DueDate, SubTotal, TaxAmt, Freight, TotalDue, ModifiedDate
        SubTotal           DECIMAL(19,4),
        TaxAmt             DECIMAL(19,4),
        Freight            DECIMAL(19,4),
        TotalDue           DECIMAL(19,4),
        ModifiedDate       TIMESTAMP
    )
    PARTITIONED BY (order_year INT, order_month INT)  -- TODO: order_year, order_month
    STORED AS PARQUET
    LOCATION '{po_path}'
""")

# TODO: jalankan MSCK REPAIR TABLE agar partisi terdeteksi
spark_hive.sql('MSCK REPAIR TABLE adventureworks.fact_purchase_orders')  # TODO: 'MSCK REPAIR TABLE adventureworks.fact_purchase_orders'

cnt = spark_hive.sql('SELECT COUNT(*) FROM adventureworks.fact_purchase_orders').collect()[0][0]
print(f'  fact_purchase_orders -> {cnt:,} rows')

spark_hive.sql('SHOW TABLES IN adventureworks').show(20)
spark_hive.stop()
print('Setup Hive External Tables selesai.')

  fact_purchase_orders -> 4,012 rows
+--------------+--------------------+-----------+
|     namespace|           tableName|isTemporary|
+--------------+--------------------+-----------+
|adventureworks|        dim_customer|      false|
|adventureworks|      dim_department|      false|
|adventureworks|        dim_employee|      false|
|adventureworks|          dim_person|      false|
|adventureworks|         dim_product|      false|
|adventureworks|dim_product_category|      false|
|adventureworks|  dim_product_subcat|      false|
|adventureworks|     dim_ship_method|      false|
|adventureworks|       dim_territory|      false|
|adventureworks|          dim_vendor|      false|
|adventureworks|fact_purchase_det...|      false|
|adventureworks|fact_purchase_orders|      false|
+--------------+--------------------+-----------+

Setup Hive External Tables selesai.


## 0. Setup SparkSession

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('CaseB_VendorPerformance') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .enableHiveSupport() \
    .getOrCreate()

print(f'Spark {spark.version} ready')
spark.sql('SHOW TABLES IN adventureworks').show(20)

Spark 3.5.0 ready
+--------------+--------------------+-----------+
|     namespace|           tableName|isTemporary|
+--------------+--------------------+-----------+
|adventureworks|        dim_customer|      false|
|adventureworks|      dim_department|      false|
|adventureworks|        dim_employee|      false|
|adventureworks|          dim_person|      false|
|adventureworks|         dim_product|      false|
|adventureworks|dim_product_category|      false|
|adventureworks|  dim_product_subcat|      false|
|adventureworks|     dim_ship_method|      false|
|adventureworks|       dim_territory|      false|
|adventureworks|          dim_vendor|      false|
|adventureworks|fact_purchase_det...|      false|
|adventureworks|fact_purchase_orders|      false|
+--------------+--------------------+-----------+



## 1. Load Data dari Hive External Tables

Semua tabel sumber sudah ada di Hive sebagai External Tables (dibuat di Bagian 0).
Baca via `spark.table('adventureworks.<nama_tabel>')`.

In [8]:
# TODO: Load semua tabel dari Hive
df_po_header  = spark.table('adventureworks.fact_purchase_orders')  # TODO: spark.table('adventureworks.fact_purchase_orders')
df_po_detail  = spark.table('adventureworks.fact_purchase_details')  # TODO: spark.table('adventureworks.fact_purchase_details')
df_vendor     = spark.table('adventureworks.dim_vendor')  # TODO: spark.table('adventureworks.dim_vendor')
df_shipmethod = spark.table('adventureworks.dim_ship_method')  # TODO: spark.table('adventureworks.dim_ship_method')
df_product    = spark.table('adventureworks.dim_product')  # TODO: spark.table('adventureworks.dim_product')

print('Row counts:')
for name, df in [
    ('fact_purchase_orders',  df_po_header),
    ('fact_purchase_details', df_po_detail),
    ('dim_vendor',            df_vendor),
    ('dim_ship_method',       df_shipmethod),
    ('dim_product',           df_product),
]:
    print(f'  {name}: {df.count():,}')


Row counts:
  fact_purchase_orders: 4,012
  fact_purchase_details: 8,845
  dim_vendor: 104
  dim_ship_method: 5
  dim_product: 504


## 2. Eksplorasi Data

In [9]:
# TODO: Eksplorasi
print('=== ShipMethod yang tersedia ===')
# TODO: tampilkan semua ShipMethod
df_shipmethod.show()

print('\n=== Sample PurchaseOrderHeader ===')
# TODO: show(5)
df_po_header.show(5)

print('\n=== Distribusi PO per ShipMethod ===')
# TODO: join header dengan shipmethod, hitung count per nama metode
df_po_header.join(df_shipmethod, df_po_header.ShipMethodID == df_shipmethod.ShipMethodID) \
    .groupBy(df_shipmethod.Name) \
    .agg(F.count('*').alias('po_count')) \
    .orderBy(F.desc('po_count')) \
    .show(20, False)

=== ShipMethod yang tersedia ===
+------------+------------------+--------+--------+--------------------+-------------------+
|ShipMethodID|              Name|ShipBase|ShipRate|             rowguid|       ModifiedDate|
+------------+------------------+--------+--------+--------------------+-------------------+
|           1|XRQ - TRUCK GROUND|  3.9500|  0.9900|6be756d9-d7be-446...|2008-04-30 00:00:00|
|           2|      ZY - EXPRESS|  9.9500|  1.9900|3455079b-f773-4dc...|2008-04-30 00:00:00|
|           3| OVERSEAS - DELUXE| 29.9500|  2.9900|22f4e461-28cf-4ac...|2008-04-30 00:00:00|
|           4|  OVERNIGHT J-FAST| 21.9500|  1.2900|107e8356-e7a8-463...|2008-04-30 00:00:00|
|           5| CARGO TRANSPORT 5|  8.9900|  1.4900|b166019a-b134-4e7...|2008-04-30 00:00:00|
+------------+------------------+--------+--------+--------------------+-------------------+


=== Sample PurchaseOrderHeader ===
+---------------+--------------+------+----------+--------+------------+-------------------+-

## 3. Extract — Filter & Join

**⚠️ Terapkan filter ShipMethod di sini!**

In [10]:
import pyspark.sql.functions as F

# TODO: Filter hanya ShipMethod CARGO TRANSPORT 5 dan OVERNIGHT J-FAST
VALID_METHODS = ['CARGO TRANSPORT 5', 'OVERNIGHT J-FAST']

# Step 1: Filter ShipMethod
df_methods_filtered = df_shipmethod.filter(F.col('Name').isin(VALID_METHODS))

# Step 2: Join PurchaseOrderHeader dengan ShipMethod yang sudah difilter
# (ini otomatis memfilter PO berdasarkan ShipMethod)
df_po_filtered = df_po_header.join(df_methods_filtered, 'ShipMethodID', 'inner')

print(f'Total PO setelah filter: {df_po_filtered.count():,}')
df_po_filtered.groupBy('Name').count().show()

# Step 3: Join dengan PO Detail dan Product
# df_po_filtered
#   JOIN df_po_detail ON PurchaseOrderID
#   JOIN df_vendor    ON VendorID == BusinessEntityID
#   JOIN df_product   ON ProductID (ambil StandardCost)
df_enriched = df_po_filtered.withColumnRenamed('Name', 'ship_method') \
    .select('PurchaseOrderID', 'VendorID', 'OrderDate', 'ShipDate', 'DueDate', 'ship_method') \
    .join(
        df_po_detail.select('PurchaseOrderID', 'ProductID', 'UnitPrice'),
        ['PurchaseOrderID'], 
        'inner'
    ) \
    .join(
        df_vendor.select(
            F.col('BusinessEntityID').alias('vendor_id'),
            F.col('Name').alias('vendor_name'),
            F.col('CreditRating')
        ),
        F.col('VendorID') == F.col('vendor_id'),
        'inner'
    ) \
    .join(
        df_product.select(
            F.col('ProductID'),
            F.col('StandardCost')
        ),
        ['ProductID'],
        'inner'
    )

# Step 4: Tambahkan kolom kalkulasi:
# - is_on_time      = ShipDate <= DueDate AND ShipDate IS NOT NULL  (1 atau 0)
# - lead_time_days  = datediff(ShipDate, OrderDate)                  (hanya jika ShipDate not null)
# - price_variance  = UnitPrice - StandardCost

# TODO: withColumn is_on_time, lead_time_days, price_variance
df_enriched = df_enriched \
    .withColumn('is_on_time', F.when(F.col('ShipDate').isNotNull() & (F.col('ShipDate') <= F.col('DueDate')), 1).otherwise(0)) \
    .withColumn('lead_time_days', F.when(F.col('ShipDate').isNotNull(), F.datediff(F.col('ShipDate'), F.col('OrderDate'))).otherwise(F.lit(None).cast('integer'))) \
    .withColumn('price_variance', F.col('UnitPrice') - F.col('StandardCost'))

print(f'Total rows enriched: {df_enriched.count():,}')
df_enriched.printSchema()

Total PO setelah filter: 2,608
+-----------------+-----+
|             Name|count|
+-----------------+-----+
|CARGO TRANSPORT 5| 1523|
| OVERNIGHT J-FAST| 1085|
+-----------------+-----+

Total rows enriched: 6,210
root
 |-- ProductID: integer (nullable = true)
 |-- PurchaseOrderID: integer (nullable = true)
 |-- VendorID: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- ShipDate: timestamp (nullable = true)
 |-- DueDate: timestamp (nullable = true)
 |-- ship_method: string (nullable = true)
 |-- UnitPrice: decimal(19,4) (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- vendor_name: string (nullable = true)
 |-- CreditRating: short (nullable = true)
 |-- StandardCost: decimal(19,4) (nullable = true)
 |-- is_on_time: integer (nullable = false)
 |-- lead_time_days: integer (nullable = true)
 |-- price_variance: decimal(20,4) (nullable = true)



## 4. Transform — Vendor Summary per ShipMethod

**Grain: 1 baris per (vendor, ship_method)**

In [11]:
# TODO: Hitung agregasi per (vendor, ship_method)
# Group by: VendorID, vendor_name, credit_rating, ship_method_name
# Agregasi:
#   - total_orders       = count(PurchaseOrderID)
#   - on_time_orders     = sum(is_on_time)
#   - on_time_rate       = ROUND(on_time_orders / total_orders * 100, 2)
#   - avg_lead_time_days = ROUND(avg(lead_time_days), 1)  ← exclude null ShipDate
#   - avg_price_variance = ROUND(avg(price_variance), 2)

df_vendor_summary = df_enriched.groupBy('VendorID', 'vendor_name', 'CreditRating', 'ship_method')\
    .agg(
        F.count('PurchaseOrderID').alias('total_orders'),
        F.sum('is_on_time').alias('on_time_orders'),
        F.round(F.avg('lead_time_days'), 1).alias('avg_lead_time_days'),
        F.round(F.avg('price_variance'), 2).alias('avg_price_variance')
    ) \
    .withColumn(
    'on_time_rate',
    F.round((F.col('on_time_orders') / F.col('total_orders')) * 100, 2)
    )



# Step 2: Hitung VendorScore
# VendorScore = (OnTimeRate * 0.6) + ((100 - ABS(AvgPriceVariance)) * 0.4)
df_vendor_summary = df_vendor_summary.withColumn(
    'vendor_score',
    F.round(
        (F.col('on_time_rate') * 0.6) + ((F.lit(100) - F.abs(F.col('avg_price_variance'))) * 0.4)
        , 2)
)

print(f'Total vendor summary rows: {df_vendor_summary.count():,}')
df_vendor_summary.orderBy(F.desc('vendor_score')).show(20, truncate=False)

Total vendor summary rows: 77
+--------+--------------------------+------------+-----------------+------------+--------------+------------------+------------------+------------+------------+
|VendorID|vendor_name               |CreditRating|ship_method      |total_orders|on_time_orders|avg_lead_time_days|avg_price_variance|on_time_rate|vendor_score|
+--------+--------------------------+------------+-----------------+------------+--------------+------------------+------------------+------------+------------+
|1620    |Lakewood Bicycle          |1           |CARGO TRANSPORT 5|51          |0             |9.0               |10.38             |0.0         |35.85       |
|1568    |Custom Frames, Inc.       |2           |CARGO TRANSPORT 5|1           |0             |9.0               |10.47             |0.0         |35.81       |
|1608    |Sport Playground          |1           |CARGO TRANSPORT 5|50          |0             |9.0               |10.82             |0.0         |35.67       |
|156

## 5. Window Functions — Ranking Vendor per ShipMethod

In [12]:
# TODO: Rank vendor per ship_method berdasarkan vendor_score
win_by_method = Window.partitionBy('ship_method').orderBy(F.desc('vendor_score'))

df_ranked = df_vendor_summary \
    .withColumn('rank_in_method', F.rank().over(win_by_method)) \
    .withColumn('score_pct_rank', F.percent_rank().over(win_by_method))

print('=== Top 5 Vendor per ShipMethod ===')
df_ranked.filter(F.col('rank_in_method') <= 5) \
         .orderBy('ship_method', 'rank_in_method') \
         .show(20, truncate=False)

# TODO: Overall rank (tanpa partisi ship_method)
# Aggregate: kalau satu vendor muncul di dua ship method, ambil avg vendor_score
df_overall = df_vendor_summary.groupBy('VendorID', 'vendor_name') \
    .agg(F.avg('vendor_score').alias('avg_vendor_score'))

win_overall = Window.orderBy(F.desc('avg_vendor_score'))
df_overall_ranked = df_overall.withColumn('overall_rank', F.rank().over(win_overall))

print('\n=== Overall Vendor Ranking ===')
df_overall_ranked.show(20, truncate=False)

=== Top 5 Vendor per ShipMethod ===


+--------+---------------------+------------+-----------------+------------+--------------+------------------+------------------+------------+------------+--------------+--------------+
|VendorID|vendor_name          |CreditRating|ship_method      |total_orders|on_time_orders|avg_lead_time_days|avg_price_variance|on_time_rate|vendor_score|rank_in_method|score_pct_rank|
+--------+---------------------+------------+-----------------+------------+--------------+------------------+------------------+------------+------------+--------------+--------------+
|1620    |Lakewood Bicycle     |1           |CARGO TRANSPORT 5|51          |0             |9.0               |10.38             |0.0         |35.85       |1             |0.0           |
|1568    |Custom Frames, Inc.  |2           |CARGO TRANSPORT 5|1           |0             |9.0               |10.47             |0.0         |35.81       |2             |0.2           |
|1608    |Sport Playground     |1           |CARGO TRANSPORT 5|50     

## 6. Load — Simpan ke Hive Curated

In [13]:
# TODO: Rename kolom agar sesuai target schema, tambah load_timestamp
spark.sql('CREATE DATABASE IF NOT EXISTS adventureworks_curated')

df_final = df_vendor_summary \
    .withColumn('load_timestamp', F.current_timestamp())

# TODO: Rename kolom agar snake_case dan sesuai spec
df_final = df_final.withColumnRenamed('VendorID', 'vendor_id') \
    .withColumnRenamed('vendor_name', 'vendor_name') \
    .withColumnRenamed('CreditRating', 'credit_rating')
# Kemudian simpan ke adventureworks_curated.fact_vendor_performance
# mode: overwrite, saveAsTable
df_final.write.mode('overwrite') \
    .saveAsTable('adventureworks_curated.fact_vendor_performance')

print('=== Verifikasi ===')
df_verify = spark.table('adventureworks_curated.fact_vendor_performance')
print(f'Total rows: {df_verify.count():,}')
df_verify.show(10, truncate=False)

=== Verifikasi ===
Total rows: 77
+---------+--------------------------+-------------+-----------------+------------+--------------+------------------+------------------+------------+------------+--------------------------+
|vendor_id|vendor_name               |credit_rating|ship_method      |total_orders|on_time_orders|avg_lead_time_days|avg_price_variance|on_time_rate|vendor_score|load_timestamp            |
+---------+--------------------------+-------------+-----------------+------------+--------------+------------------+------------------+------------+------------+--------------------------+
|1536     |Cruger Bike Company       |1            |OVERNIGHT J-FAST |210         |0             |9.0               |41.77             |0.0         |23.29       |2026-05-15 07:02:52.293671|
|1686     |Pro Sport Industries      |1            |CARGO TRANSPORT 5|17          |0             |9.0               |44.89             |0.0         |22.04       |2026-05-15 07:02:52.293671|
|1554     |WestA

## 7. Analytic Queries

**Wajib: minimal 3 query yang menjawab business question**

In [20]:
# ── Query 1 (WAJIB): Ranking vendor berdasarkan VendorScore ────────────────
# Tampilkan: vendor_name, ship_method, on_time_rate, avg_price_variance, vendor_score
# Sort: desc vendor_score

print('=== Query 1: Vendor Ranking by VendorScore ===')
spark.sql("""
    -- TODO: tulis query SQL kamu di sini
    SELECT vendor_name, ship_method, on_time_rate, avg_price_variance, vendor_score
    FROM adventureworks_curated.fact_vendor_performance
    ORDER BY vendor_score DESC;
""").show()


=== Query 1: Vendor Ranking by VendorScore ===
+--------------------+-----------------+------------+------------------+------------+
|         vendor_name|      ship_method|on_time_rate|avg_price_variance|vendor_score|
+--------------------+-----------------+------------+------------------+------------+
|    Lakewood Bicycle|CARGO TRANSPORT 5|         0.0|             10.38|       35.85|
| Custom Frames, Inc.|CARGO TRANSPORT 5|         0.0|             10.47|       35.81|
|    Sport Playground|CARGO TRANSPORT 5|         0.0|             10.82|       35.67|
| Capital Road Cycles|CARGO TRANSPORT 5|         0.0|             12.43|       35.03|
|Chicago City Saddles| OVERNIGHT J-FAST|         0.0|             13.91|       34.44|
|Chicago City Saddles|CARGO TRANSPORT 5|         0.0|             14.90|       34.04|
|        Trikes, Inc.| OVERNIGHT J-FAST|         0.0|             16.57|       33.37|
|Greenwood Athleti...|CARGO TRANSPORT 5|         0.0|             17.12|       33.15|
|     M

In [21]:
# ── Query 2 (WAJIB): Perbandingan OnTimeRate per ShipMethod ────────────────
# Tampilkan: ship_method, avg_on_time_rate, total_vendors, total_orders

print('=== Query 2: OnTimeRate per ShipMethod ===')
# TODO
spark.sql("""
    SELECT ship_method, ROUND(AVG(on_time_rate), 2) AS avg_on_time_rate, COUNT(DISTINCT vendor_id) AS total_vendors, SUM(total_orders) AS total_orders
    FROM adventureworks_curated.fact_vendor_performance
    GROUP BY ship_method;
""").show()

=== Query 2: OnTimeRate per ShipMethod ===
+-----------------+----------------+-------------+------------+
|      ship_method|avg_on_time_rate|total_vendors|total_orders|
+-----------------+----------------+-------------+------------+
|CARGO TRANSPORT 5|             0.0|           45|        3225|
| OVERNIGHT J-FAST|             0.0|           32|        2985|
+-----------------+----------------+-------------+------------+



In [25]:
# ── Query 3 (WAJIB): Vendor dengan AvgPriceVariance negatif ───────────────
# Artinya: beli lebih murah dari StandardCost (menguntungkan perusahaan)
# Tampilkan: vendor_name, ship_method, avg_price_variance, on_time_rate, vendor_score

print('=== Query 3: Vendor dengan Harga Lebih Murah dari StandardCost ===')
# TODO
spark.sql("""
    SELECT vendor_name, ship_method, avg_price_variance, on_time_rate, vendor_score
    FROM adventureworks_curated.fact_vendor_performance
    WHERE avg_price_variance < 0;
""").show()

=== Query 3: Vendor dengan Harga Lebih Murah dari StandardCost ===
+-----------+-----------+------------------+------------+------------+
|vendor_name|ship_method|avg_price_variance|on_time_rate|vendor_score|
+-----------+-----------+------------------+------------+------------+
+-----------+-----------+------------------+------------+------------+



In [ ]:
# ── Query 4 (BONUS): Korelasi CreditRating dan OnTimeRate ──────────────────
# Tampilkan: credit_rating, avg_on_time_rate, avg_vendor_score, count_vendors
# Insight: apakah vendor dengan credit rating tinggi lebih on-time?

print('=== Query 4 (Bonus): CreditRating vs OnTimeRate ===')
# TODO

In [ ]:
# ── Query 5 (BONUS): Top 3 vendor per ShipMethod berdasarkan TotalOrders ───
# Gunakan window function rank() partitionBy ship_method, orderBy desc total_orders

print('=== Query 5 (Bonus): Top 3 Vendor per ShipMethod by TotalOrders ===')
# TODO

In [ ]:
spark.stop()
print('Done!')